# 10주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 과제 내용
전이학습 모델과 직접 구현한 CNN의 학습 시간과 결과를 비교합니다.

## 제출 방법
모든 셀을 실행해 출력과 그래프가 보이는 상태로 저장한 뒤 `week10_학번_이름.ipynb` 로 제출합니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import pandas as pd
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# 고정 조건 — 바꾸지 마세요
EPOCHS, BATCH, SEED, IMG_SIZE = 10, 16, 42, 224

NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), NORM,
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(), NORM,
])

raw_train = datasets.Flowers102("./data", split="train", download=True, transform=train_tf)
raw_val   = datasets.Flowers102("./data", split="val",   download=True, transform=eval_tf)

KEEP = [0, 1, 2, 3, 4]
NAMES = ["pink primrose", "hard-leaved orchid", "canterbury bells", "sweet pea", "english marigold"]

class Subset5(torch.utils.data.Dataset):
    def __init__(self, base):
        self.base = base
        self.idx = [i for i, lb in enumerate(base._labels) if lb in KEEP]
        self.remap = {c: i for i, c in enumerate(KEEP)}
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        x, y = self.base[self.idx[i]]
        return x, self.remap[y]

train_loader = DataLoader(Subset5(raw_train), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(Subset5(raw_val),   batch_size=32, shuffle=False)
N_CLASSES = len(KEEP)
print(f"학습 {len(train_loader.dataset)}장 · 검증 {len(val_loader.dataset)}장 · 클래스 {N_CLASSES}개")


## 공통 함수
아래를 그대로 쓰고 모델만 바꿔 실험하세요.


In [ ]:
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


def run(model, lr, epochs=EPOCHS, seed=SEED):
    torch.manual_seed(seed)
    model = model.to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    start = time.time()
    hist = []
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append(evaluate(model, val_loader))
        if epoch % 5 == 0:
            print(f"  epoch {epoch}  검증 {hist[-1][1]:.4f}")
    return {"model": model, "hist": hist, "time": time.time() - start,
            "train_params": sum(p.numel() for p in model.parameters() if p.requires_grad)}


## 문제 1. 직접 구현한 CNN (25점)
8·9주차에서 배운 것을 써서 CNN을 직접 작성하고 학습하세요.

- Conv 층 3개 이상
- BatchNorm 또는 Dropout 중 하나 이상 적용
- 입력은 (3, 224, 224) 컬러 이미지
- 왜 그렇게 구성했는지 주석으로 적으세요


In [ ]:
# 답안


## 문제 2. 전이학습 모델 (25점)
`resnet18` 을 불러와 마지막 계층을 교체하고 학습하세요.

- `in_features` 를 하드코딩하지 말고 읽어서 쓰세요
- 특징 추출 부분은 고정합니다
- 학습 파라미터 수를 출력하세요


In [ ]:
# 답안


## 문제 3. 층 고정 방식 비교 (20점)
아래 세 가지를 비교하세요.

1. 마지막 계층만 학습 (lr=1e-3)
2. `layer4` + 마지막 계층 학습 (lr=1e-4)
3. 전체 미세조정 (lr=1e-4)

각 방식의 학습 파라미터 수도 함께 적으세요.


In [ ]:
# 답안


## 문제 4. 비교표와 학습 곡선 (20점)
모든 모델을 하나의 표로 정리하세요.

| 모델 | 학습 파라미터 | 최종 검증 정확도 | 최고 검증 정확도 | 학습 시간 |

학습 곡선을 겹쳐 그린 그림도 함께 넣으세요.


In [ ]:
# 답안


## 문제 5. 예측 결과 확인과 해석 (10점)
가장 좋은 모델로 검증 이미지 12장의 예측을 그리세요. 각 이미지에 **예측 클래스 · 예측 확률 · 실제 레이블**을 표시하고, 틀린 이미지도 따로 모아 보세요.

그리고 아래 세 가지를 각각 두세 줄로 적으세요.

1. 적은 데이터에서 전이학습이 유리한 이유
2. 층을 많이 풀수록 좋아지는가 — 이 데이터에서는 어땠나
3. 학습 시간과 정확도를 함께 볼 때 어느 모델을 고를 것인가


In [ ]:
# 예측 결과 확인 코드


**답:**

1.

2.

3.
